# LISTA-CP-KAN: Adaptive Thresholds via Kolmogorov-Arnold Networks

This notebook trains and evaluates LISTA-CP with KAN-based adaptive thresholds.

**Steps:**
1. Install dependencies
2. Clone repo / upload files
3. Train baseline (16-stage greedy)
4. Train KAN variants (1norm and 3norm)
5. Evaluate and generate plots

In [ ]:
# Cell 1: Install dependencies
!pip install torch numpy matplotlib scikit-learn pyyaml pandas pykan==0.2.8 -q

In [ ]:
# Cell 2: Upload project files or clone from repo
# Option A: If files are in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/cs754_research_project

# Option B: If uploading directly, place all .py files in /content/project/
import os
PROJECT_DIR = '/content/project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'GPU available: {os.popen("nvidia-smi --query-gpu=name --format=csv,noheader").read().strip()}')

In [ ]:
# Cell 3: Verify imports work
import torch
import numpy as np
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

from problem_setup import random_A, gen_samples_torch, compute_nmse_db_torch
from lista_cp_torch import LISTACPTorch
from lista_cp_kan import LISTACPKan
from kan_theta_predictor import KANThetaPredictor
print('All imports successful.')

## Step 1: Train Baseline (16-stage greedy)

This takes ~15-25 minutes on a T4 GPU. Each stage trains W_k and theta_k for one layer with all earlier layers frozen.

In [ ]:
# Cell 4: Train baseline
!python train_baseline.py

In [ ]:
# Cell 5: Verify baseline checkpoint
ckpt = torch.load('lista_cp_baseline.pth', map_location='cpu', weights_only=False)
print(f"Baseline trained thetas: {[round(v, 5) for v in ckpt['thetas']]}")
print(f"Test NMSE per layer: {[round(v, 2) for v in ckpt['test_nmse_per_layer']]}")
print(f"Final layer 16 NMSE: {ckpt['test_nmse_per_layer'][-1]:.2f} dB")

## Step 2: Train KAN Variants

Two-phase training:
- **Phase A** (50 epochs): KAN parameters only, W frozen
- **Phase B** (50 epochs): Joint fine-tuning of W and KAN

Each variant takes ~10-15 minutes on a T4.

In [ ]:
# Cell 6: Train 1norm variant
!python train_kan.py --variant 1norm

In [ ]:
# Cell 7: Train 3norm variant
!python train_kan.py --variant 3norm

## Step 3: Evaluation

Generates NMSE curves, parameter counts, adaptivity checks, and spline visualizations.

In [ ]:
# Cell 8: Run full evaluation
!python evaluate.py

In [ ]:
# Cell 9: Display result plots
import matplotlib.pyplot as plt
from IPython.display import display, Image
import glob

plot_files = sorted(glob.glob('results/*.png'))
for pf in plot_files:
    print(f'\n--- {os.path.basename(pf)} ---')
    display(Image(filename=pf))

In [ ]:
# Cell 10: Download results (optional)
# from google.colab import files
# import shutil
# shutil.make_archive('results', 'zip', '.', 'results')
# files.download('results.zip')
# files.download('lista_cp_baseline.pth')
# files.download('lista_cp_kan1_final.pth')
# files.download('lista_cp_kan3_final.pth')